# All GL code revenue forecasts
Run from top to bottom. Settings, models, forecasts, validation, exports, and charts are separate.
All revenue Types are included. GL codes 42, 4860, and 4870 are excluded.
Every remaining GL appears in the output, including codes with insufficient or stale data.

| Method | Description |
|---|---|
| Linear | Straight-line regression on recent annual revenue |
| Holt | Exponential smoothing with a damped trend |
| ML (Random forest) | Two annual revenue lags and time; recursive predictions |
| Monte Carlo | 10,000 paths using sampled recent annual dollar changes |
| Naive baseline | Last recorded annual revenue repeated |

Monte Carlo P10/P90 describe the middle 80% of simulated outcomes, not a guaranteed revenue range.

## 1. Settings and imports
Keep the same data, settings, Python version, and package versions to reproduce a run.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import warnings
from html import escape
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.holtwinters import Holt
import plotly.graph_objects as go

LAST_COMPLETE_FY = 2026
FORECAST_HORIZON = 3
EXCLUDED_GL_CODES = [42, 4860, 4870]
LOOKBACK = 10
MIN_TRAIN_YEARS = 4
BACKTEST_ORIGINS = 3
N_SIMULATIONS = 10000
MC_CHANGE_WINDOW = 5
RF_TREES = 100
RF_MAX_DEPTH = 3
SEED = 2026
# Alternatives: 'zero_fill' or 'carry_forward'. Do not change without a data assumption.
MISSING_YEAR_POLICY = 'unavailable'
MODELS = ['Linear', 'Holt', 'ML (Random forest)', 'Monte Carlo', 'Naive baseline']


## 2. Read the source data
Use the same candidate-path approach as the earlier notebooks. All Types, including MAJOR, are retained.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = RAW_DIR / 'processed'
candidate_paths = [
    RAW_DIR / 'dawn_revenue_accounts_combined_history.csv',
    PROJECT_ROOT.parent / 'data' / 'dawn_revenue_accounts_combined_history.csv',
    PROJECT_ROOT.parent / 'dawn_aggregate' / 'data' / 'dawn_revenue_accounts_combined_history.csv',
    PROCESSED_DIR / 'dawn_revenue_accounts_combined_history.csv',
]
input_csv = next((path for path in candidate_paths if path.exists()), None)
if input_csv is None:
    raise FileNotFoundError(f'Data file not found. Checked: {candidate_paths}')
input_csv = input_csv.resolve()
project_root = next(parent for parent in input_csv.parents if (parent / 'notebook').is_dir())
result_dir = project_root / 'result'
plot_dir = result_dir / 'allGLcode_forecast_plots'
input_sha256 = hashlib.sha256(input_csv.read_bytes()).hexdigest()
df_all = pd.read_csv(input_csv)
required_columns = ['GL Number', 'fy', 'Amount', 'description', 'Type']
if not set(required_columns).issubset(df_all.columns):
    raise ValueError(f'Required columns: {required_columns}')
for column in ['GL Number', 'fy', 'Amount']:
    df_all[column] = pd.to_numeric(df_all[column], errors='raise')
if df_all[['GL Number', 'fy', 'Amount']].isna().any().any():
    raise ValueError('Missing GL, fiscal year, or revenue values require review.')
df_all_filtered = df_all.loc[
    ~df_all['GL Number'].isin(EXCLUDED_GL_CODES) & df_all['fy'].le(LAST_COMPLETE_FY)
].copy()
annual_revenue = (
    df_all_filtered.groupby(['GL Number', 'fy'], as_index=False)
    .agg(revenue=('Amount', 'sum'), description=('description', 'first'))
    .sort_values(['GL Number', 'fy'])
)
all_gl_codes = sorted(int(gl) for gl in annual_revenue['GL Number'].unique())
future_fys = np.arange(LAST_COMPLETE_FY + 1, LAST_COMPLETE_FY + 1 + FORECAST_HORIZON)
print(f'{len(all_gl_codes)} GL codes; forecast FY {future_fys[0]}-{future_fys[-1]}.')
display(annual_revenue.head())


## 3. Available history and data status
By default, missing years are not treated as zero. Models use the latest uninterrupted history.
A GL without FY 2026 records remains in the CSV with unavailable forecasts and a review verdict.
Current GLs with short histories get only methods that have enough observations.
Negative net revenue is preserved: a model trained on negative values has no zero floor.

In [ ]:
if MISSING_YEAR_POLICY not in {'unavailable', 'zero_fill', 'carry_forward'}:
    raise ValueError('Unknown MISSING_YEAR_POLICY.')
histories = {}
quality_rows = []
for gl in all_gl_codes:
    actual = annual_revenue.loc[annual_revenue['GL Number'].eq(gl)].copy()
    recorded_end = int(actual['fy'].iloc[-1])
    years = actual['fy'].to_numpy(dtype=int)
    missing = sorted(set(range(int(years[0]), LAST_COMPLETE_FY + 1)) - set(years))
    history = actual.copy()
    if MISSING_YEAR_POLICY == 'zero_fill':
        history = history.set_index('fy').reindex(range(int(years[0]), LAST_COMPLETE_FY + 1))
        history['revenue'] = history['revenue'].fillna(0)
        history['GL Number'] = gl
        history['description'] = actual['description'].iloc[-1]
        history = history.reset_index(names='fy')
    else:
        breaks = np.flatnonzero(np.diff(years) != 1)
        if len(breaks):
            history = history.iloc[breaks[-1] + 1:]
    if not np.isfinite(history['revenue']).all():
        raise ValueError(f'GL {gl}: non-finite annual revenue.')
    histories[gl] = history
    stale = recorded_end < LAST_COMPLETE_FY
    status = 'CURRENT'
    if stale:
        status = 'STALE: no records in cutoff FY'
    elif len(history) < MIN_TRAIN_YEARS:
        status = 'SHORT HISTORY: some methods unavailable'
    quality_rows.append({
        'GL Number': gl, 'Description': actual['description'].iloc[-1],
        'First recorded FY': int(years[0]), 'Last recorded FY': recorded_end,
        'Recorded years': len(actual), 'Model history start FY': int(history['fy'].iloc[0]),
        'Model history years': len(history), 'Final training years': min(len(history), LOOKBACK),
        'Missing fiscal years': ', '.join(map(str, missing)),
        'Negative annual totals': int((actual['revenue'] < 0).sum()),
        'Data status': status, 'Missing-year policy': MISSING_YEAR_POLICY,
    })
data_quality_table = pd.DataFrame(quality_rows)
quality_lookup = data_quality_table.set_index('GL Number')
display(data_quality_table)


## 4. Model functions
Hyperparameters and seeds match the selected-GL workflow. Each fit receives only past data.
Linear needs 2 observations, Holt and random forest need 4, Monte Carlo needs 2, and the baseline needs 1.
Constant series use an exact constant forecast, avoiding unnecessary optimizer warnings.

In [ ]:
fit_diagnostics = []


def fit_forecasts(values, horizon, seed, context):
    y = np.asarray(values, dtype=float)[-LOOKBACK:]
    scale = max(float(np.max(np.abs(y))), 1.0)
    z = y / scale
    floor_at_zero = not (y < 0).any()

    def apply_floor(values):
        values = np.asarray(values, dtype=float)
        return np.maximum(values, 0) if floor_at_zero else values

    predictions = {model: np.full(horizon, np.nan) for model in MODELS}
    predictions['Naive baseline'] = np.repeat(y[-1], horizon)
    low, high = np.full(horizon, np.nan), np.full(horizon, np.nan)
    minimum = {'Linear': 2, 'Holt': MIN_TRAIN_YEARS,
               'ML (Random forest)': MIN_TRAIN_YEARS, 'Monte Carlo': 2}
    for model, needed in minimum.items():
        if len(y) < needed:
            continue
        try:
            if np.all(y == y[0]):
                predictions[model] = np.repeat(y[-1], horizon)
                if model == 'Monte Carlo':
                    low, high = predictions[model].copy(), predictions[model].copy()
                continue
            if model == 'Linear':
                reg = LinearRegression().fit(np.arange(len(y)).reshape(-1, 1), z)
                pred = reg.predict(np.arange(len(y), len(y) + horizon).reshape(-1, 1)) * scale
            elif model == 'Holt':
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter('always')
                    fit = Holt(z, damped_trend=True, initialization_method='estimated').fit(optimized=True)
                for warning in caught:
                    fit_diagnostics.append({'Context': context, 'Model': model, 'Message': str(warning.message)})
                pred = np.asarray(fit.forecast(horizon)) * scale
            elif model == 'ML (Random forest)':
                features = np.array([[z[i-1], z[i-2], i] for i in range(2, len(z))])
                fit = RandomForestRegressor(n_estimators=RF_TREES, max_depth=RF_MAX_DEPTH,
                    min_samples_leaf=1, random_state=seed, n_jobs=1).fit(features, z[2:])
                extended = z.tolist()
                for _ in range(horizon):
                    extended.append(float(fit.predict([[extended[-1], extended[-2], len(extended)]])[0]))
                pred = np.asarray(extended[-horizon:]) * scale
            else:
                rng = np.random.default_rng(seed)
                changes = np.diff(y)[-MC_CHANGE_WINDOW:]
                draws = rng.choice(changes, size=(N_SIMULATIONS, horizon), replace=True)
                paths = np.empty_like(draws)
                previous = np.full(N_SIMULATIONS, y[-1])
                for h in range(horizon):
                    previous = apply_floor(previous + draws[:, h])
                    paths[:, h] = previous
                pred = paths.mean(axis=0)
                low, high = np.quantile(paths, 0.1, axis=0), np.quantile(paths, 0.9, axis=0)
            if not np.isfinite(pred).all():
                raise ValueError('Model returned non-finite forecasts.')
            predictions[model] = apply_floor(pred)
        except (ValueError, RuntimeError, FloatingPointError, np.linalg.LinAlgError) as error:
            fit_diagnostics.append({'Context': context, 'Model': model, 'Message': str(error)})
    return predictions, low, high


## 5. Historical backtests
Use up to three recent origins with three unseen years each. If too short, use the last year
as a limited holdout. No future actuals enter training. Stale GLs can have historical tests,
but those tests do not establish current forecasting suitability.

In [ ]:
fit_diagnostics = []
backtest_rows = []
for position, (gl, history) in enumerate(histories.items(), start=1):
    years = history['fy'].to_numpy(dtype=int)
    values = history['revenue'].to_numpy(dtype=float)
    origins = list(range(MIN_TRAIN_YEARS, len(values) - FORECAST_HORIZON + 1))[-BACKTEST_ORIGINS:]
    if not origins and len(values) >= 2:
        origins = [len(values) - 1]
    for cutoff in origins:
        origin = int(years[cutoff - 1])
        horizon = min(FORECAST_HORIZON, len(values) - cutoff)
        predictions, low, high = fit_forecasts(values[:cutoff], horizon, SEED + gl + origin,
                                              f'GL {gl}, origin FY {origin}')
        for model, predicted in predictions.items():
            for h in range(horizon):
                backtest_rows.append({
                    'GL Number': gl, 'Model': model, 'Origin FY': origin,
                    'Target FY': int(years[cutoff+h]), 'Horizon': h+1,
                    'Actual': values[cutoff+h], 'Predicted': predicted[h],
                    'MC P10': low[h] if model == 'Monte Carlo' else np.nan,
                    'MC P90': high[h] if model == 'Monte Carlo' else np.nan,
                })
    if position % 25 == 0:
        print(f'Backtested {position}/{len(histories)} GL codes.')
backtest_predictions = pd.DataFrame(backtest_rows, columns=[
    'GL Number', 'Model', 'Origin FY', 'Target FY', 'Horizon', 'Actual', 'Predicted', 'MC P10', 'MC P90'])
print('Backtesting complete.')


## 6. Forecast FY 2027-2029
Blank forecast values mean the method could not be supported by the available data.
A stale-data carry-forward option supplies only the baseline, never a disguised fitted model.

In [ ]:
forecast_rows = []
fit_diagnostics = [item for item in fit_diagnostics if ', final FY ' not in item['Context']]
for gl, history in histories.items():
    stale = quality_lookup.loc[gl, 'Last recorded FY'] < LAST_COMPLETE_FY
    values = history['revenue'].to_numpy(dtype=float)
    if stale and MISSING_YEAR_POLICY != 'zero_fill':
        predictions = {model: np.full(FORECAST_HORIZON, np.nan) for model in MODELS}
        low, high = np.full(FORECAST_HORIZON, np.nan), np.full(FORECAST_HORIZON, np.nan)
        if MISSING_YEAR_POLICY == 'carry_forward':
            predictions['Naive baseline'] = np.repeat(values[-1], FORECAST_HORIZON)
    else:
        predictions, low, high = fit_forecasts(values, FORECAST_HORIZON, SEED + gl,
                                              f'GL {gl}, final FY {LAST_COMPLETE_FY}')
    for h, year in enumerate(future_fys):
        forecast_rows.append({
            'GL Number': gl, 'Description': quality_lookup.loc[gl, 'Description'], 'FY': int(year),
            'Last actual FY': int(quality_lookup.loc[gl, 'Last recorded FY']),
            'Last actual': annual_revenue.loc[annual_revenue['GL Number'].eq(gl), 'revenue'].iloc[-1],
            **{model: predicted[h] for model, predicted in predictions.items()},
            'MC P10': low[h], 'MC P90': high[h], 'Data status': quality_lookup.loc[gl, 'Data status'],
        })
forecast_table = pd.DataFrame(forecast_rows)
print(f'Created {len(forecast_table)} GL/year rows, including explicitly unavailable forecasts.')


## 7. Validation metrics and verdicts
MAE/RMSE are dollar errors; WAPE is total absolute error / total absolute actual revenue.
Positive MAE skill means improvement over the baseline on the same forecast cases.

| Verdict | Meaning |
|---|---|
| IMPROVES ON BASELINE | Lower MAE, at least two origins, all three horizons, and no missing model fits |
| DOES NOT IMPROVE ON BASELINE | Sufficient backtest coverage, but no MAE improvement |
| LIMITED VALIDATION | Too few origins/horizons to assess three-year accuracy |
| NO VALIDATION | No supported out-of-sample forecasts |
| REVIEW FIT WARNINGS | Model fitting emitted a warning or failure |
| REVIEW STALE DATA | No actual record in the cutoff FY; overrides a historical model verdict |

These are evidence-based comparisons, not a guarantee of forecast accuracy or formal pass/fail thresholds.

In [ ]:
def metrics(group):
    valid = group.dropna(subset=['Actual', 'Predicted'])
    if valid.empty:
        return {'Forecast cases': 0, 'Origins': 0, 'Max horizon': 0, 'MAE ($)': np.nan,
                'RMSE ($)': np.nan, 'WAPE (%)': np.nan, 'Bias ($)': np.nan, 'MC 80% coverage (%)': np.nan}
    errors = valid['Predicted'] - valid['Actual']
    denominator = valid['Actual'].abs().sum()
    return {
        'Forecast cases': len(valid), 'Origins': valid['Origin FY'].nunique(),
        'Max horizon': int(valid['Horizon'].max()), 'MAE ($)': errors.abs().mean(),
        'RMSE ($)': np.sqrt(np.mean(errors ** 2)),
        'WAPE (%)': 100 * errors.abs().sum() / denominator if denominator else np.nan,
        'Bias ($)': errors.mean(),
        'MC 80% coverage (%)': (100 * valid['Actual'].between(valid['MC P10'], valid['MC P90']).mean()
                               if valid['Model'].eq('Monte Carlo').all() else np.nan),
    }


validation_rows = []
for gl in all_gl_codes:
    baseline = backtest_predictions.loc[backtest_predictions['GL Number'].eq(gl)
                                        & backtest_predictions['Model'].eq('Naive baseline')]
    for model in MODELS:
        group = backtest_predictions.loc[backtest_predictions['GL Number'].eq(gl)
                                         & backtest_predictions['Model'].eq(model)]
        measured = metrics(group)
        matched = group.dropna(subset=['Predicted']).merge(
            baseline[['Origin FY', 'Target FY', 'Horizon', 'Predicted']],
            on=['Origin FY', 'Target FY', 'Horizon'], suffixes=('', '_baseline'))
        base_mae = (matched['Predicted_baseline'] - matched['Actual']).abs().mean()
        skill = 100 * (1 - measured['MAE ($)'] / base_mae) if base_mae > 0 else np.nan
        enough = (measured['Origins'] >= 2 and measured['Max horizon'] == FORECAST_HORIZON
                  and measured['Forecast cases'] == len(baseline))
        warned = any(item['Context'].startswith(f'GL {gl},') and item['Model'] == model
                     for item in fit_diagnostics)
        if measured['Forecast cases'] == 0:
            verdict = 'NO VALIDATION'
        elif not enough:
            verdict = 'LIMITED VALIDATION'
        elif model == 'Naive baseline':
            verdict = 'BASELINE BENCHMARK'
        elif measured['MAE ($)'] < base_mae and not np.isclose(measured['MAE ($)'], base_mae):
            verdict = 'IMPROVES ON BASELINE'
        else:
            verdict = 'DOES NOT IMPROVE ON BASELINE'
        if warned:
            verdict = 'REVIEW FIT WARNINGS'
        if quality_lookup.loc[gl, 'Last recorded FY'] < LAST_COMPLETE_FY:
            verdict = 'REVIEW STALE DATA'
        validation_rows.append({'GL Number': gl, 'Model': model, **measured,
            'Baseline MAE on matched cases ($)': base_mae, 'MAE skill vs naive (%)': skill,
            'Comparable cases': measured['Forecast cases'] > 0 and measured['Forecast cases'] == len(baseline),
            'Data status': quality_lookup.loc[gl, 'Data status'], 'Verdict': verdict})
validation_table = pd.DataFrame(validation_rows)
validation_by_horizon = pd.DataFrame([
    {'GL Number': gl, 'Model': model, 'Horizon': horizon, **metrics(group)}
    for (gl, model, horizon), group in backtest_predictions.groupby(['GL Number', 'Model', 'Horizon'])
])
gl_verdict_rows = []
for gl in all_gl_codes:
    available = forecast_table.loc[forecast_table['GL Number'].eq(gl)].iloc[0]
    candidates = validation_table.loc[validation_table['GL Number'].eq(gl)
        & validation_table['Comparable cases'] & validation_table['MAE ($)'].notna()].copy()
    candidates = candidates.loc[candidates['Model'].map(lambda model: pd.notna(available[model]))]
    candidates = candidates.loc[~candidates['Verdict'].eq('REVIEW FIT WARNINGS')]
    if candidates.empty:
        chosen = 'Naive baseline' if pd.notna(available['Naive baseline']) else None
        verdict = 'UNVALIDATED BASELINE ONLY' if chosen else 'UNAVAILABLE: resolve missing cutoff-year data'
    else:
        winner = candidates.sort_values('MAE ($)', kind='stable').iloc[0]
        chosen, verdict = winner['Model'], winner['Verdict']
    if quality_lookup.loc[gl, 'Last recorded FY'] < LAST_COMPLETE_FY:
        verdict = 'REVIEW STALE DATA: ' + ('scenario only' if chosen else 'forecast unavailable')
    gl_verdict_rows.append({'GL Number': gl, 'Selected model': chosen, 'GL verdict': verdict})
gl_verdict_table = data_quality_table.merge(pd.DataFrame(gl_verdict_rows), on='GL Number')
forecast_table = forecast_table.merge(pd.DataFrame(gl_verdict_rows), on='GL Number')
forecast_table['Selected-model forecast'] = [row[row['Selected model']] if pd.notna(row['Selected model'])
                                            else np.nan for _, row in forecast_table.iterrows()]
display(validation_table.style.format(precision=2, na_rep='Unavailable'))


## 8. Per-GL verdict table
Model selection is descriptive on these backtests, not independently tested. A baseline can win.

In [ ]:
display(gl_verdict_table)


## 9. Forecast values
Each row is one GL and future fiscal year. All five methods and the Monte Carlo bounds are retained.

In [ ]:
money_columns = ['Last actual'] + MODELS + ['MC P10', 'MC P90', 'Selected-model forecast']
display(forecast_table.style.format({column: '${:,.2f}' for column in money_columns}, na_rep='Unavailable'))


## 10. Model comparison on common cases
Pooled metrics use only GL/origin/horizon cases where all five models produced a forecast.
Large revenue sources have more influence on pooled dollar errors. Use the per-GL verdicts too.

In [ ]:
keys = ['GL Number', 'Origin FY', 'Target FY', 'Horizon']
complete = backtest_predictions.groupby(keys)['Predicted'].apply(lambda values: values.notna().sum() == len(MODELS))
common_keys = complete.loc[complete].reset_index()[keys]
common_cases = backtest_predictions.merge(common_keys, on=keys)
model_comparison = pd.DataFrame([
    {'Model': model, **metrics(common_cases.loc[common_cases['Model'].eq(model)])} for model in MODELS])
display(model_comparison.style.format(precision=2, na_rep='Unavailable'))


## 11. Export CSVs and reproducibility records
Files go under `result`. If a CSV is open and locked, save an `_updated.csv` copy and record its name.

In [ ]:
result_dir.mkdir(parents=True, exist_ok=True)
plot_dir.mkdir(parents=True, exist_ok=True)
exported_files = {}
for filename, table in {
    'allGLcode_forecasts.csv': forecast_table,
    'allGLcode_model_validation.csv': validation_table,
    'allGLcode_verdicts.csv': gl_verdict_table,
    'allGLcode_validation_by_horizon.csv': validation_by_horizon,
    'allGLcode_backtest_predictions.csv': backtest_predictions,
    'allGLcode_model_comparison.csv': model_comparison,
    'allGLcode_data_quality.csv': data_quality_table,
    'allGLcode_fit_diagnostics.csv': pd.DataFrame(fit_diagnostics, columns=['Context', 'Model', 'Message']),
}.items():
    destination = result_dir / filename
    try:
        table.to_csv(destination, index=False)
    except PermissionError:
        destination = destination.with_name(destination.stem + '_updated.csv')
        table.to_csv(destination, index=False)
    exported_files[filename] = destination.name
packages = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'statsmodels', 'plotly', 'ipython', 'jinja2']
versions = {package: importlib.metadata.version(package) for package in packages}
run_settings = {
    'input_file': str(input_csv.relative_to(project_root)), 'input_sha256': input_sha256,
    'last_complete_fy': LAST_COMPLETE_FY, 'forecast_horizon': FORECAST_HORIZON,
    'excluded_gl_codes': EXCLUDED_GL_CODES, 'revenue_types': 'ALL',
    'missing_year_policy': MISSING_YEAR_POLICY, 'lookback': LOOKBACK,
    'min_train_years': MIN_TRAIN_YEARS, 'backtest_origins': BACKTEST_ORIGINS,
    'seed': SEED, 'n_simulations': N_SIMULATIONS, 'mc_change_window': MC_CHANGE_WINDOW,
    'rf_trees': RF_TREES, 'rf_max_depth': RF_MAX_DEPTH, 'rf_min_samples_leaf': 1, 'rf_n_jobs': 1,
    'holt_damped_trend': True, 'holt_initialization': 'estimated',
    'forecast_floor': 'Zero only when the training window contains no negative revenue',
    'python_version': platform.python_version(), 'package_versions': versions,
    'exported_files': exported_files, 'gl_codes': all_gl_codes,
}
(result_dir / 'allGLcode_run_settings.json').write_text(json.dumps(run_settings, indent=2), encoding='utf-8')
(result_dir / 'allGLcode_requirements.txt').write_text(
    '\n'.join(f'{package}=={version}' for package, version in versions.items()) + '\n', encoding='utf-8')
print(f'CSV files saved in {result_dir}')
display(pd.DataFrame(exported_files.items(), columns=['Table', 'Saved filename']))


## 12. Historical-versus-forecast chart function
One interactive chart per GL. Actual observations include earlier history; gaps are displayed as gaps.
Forecasts use the latest continuous block. An annotation explains unavailable forecasts.

In [ ]:
MODEL_COLORS = {'Linear': '#0072B2', 'Holt': '#D55E00', 'ML (Random forest)': '#009E73',
                'Monte Carlo': '#CC79A7', 'Naive baseline': '#888888'}


def plot_gl_forecast(gl):
    actual = annual_revenue.loc[annual_revenue['GL Number'].eq(gl)]
    forecast = forecast_table.loc[forecast_table['GL Number'].eq(gl)]
    observed = actual.set_index('fy')['revenue'].reindex(range(int(actual['fy'].min()), LAST_COMPLETE_FY + 1))
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=observed.index, y=observed, name='Actual', mode='lines+markers',
                            connectgaps=False, line=dict(color='#222222')))
    if forecast['MC P10'].notna().any():
        fig.add_trace(go.Scatter(x=forecast['FY'], y=forecast['MC P10'], mode='lines',
            line=dict(width=0), legendgroup='MC range', showlegend=False, hoverinfo='skip'))
        fig.add_trace(go.Scatter(x=forecast['FY'], y=forecast['MC P90'], mode='lines', line=dict(width=0),
            fill='tonexty', fillcolor='rgba(204,121,167,0.17)', name='MC P10-P90', legendgroup='MC range',
            customdata=forecast[['MC P10']],
            hovertemplate='FY %{x}<br>MC range: $%{customdata[0]:,.0f} to $%{y:,.0f}<extra></extra>'))
    for model in MODELS:
        if forecast[model].isna().all():
            continue
        anchor = observed.loc[LAST_COMPLETE_FY]
        if pd.isna(anchor) and MISSING_YEAR_POLICY == 'zero_fill':
            anchor = 0
        fig.add_trace(go.Scatter(x=[LAST_COMPLETE_FY, *forecast['FY']],
            y=[anchor, *forecast[model]], name=model, mode='lines+markers',
            line=dict(color=MODEL_COLORS[model], dash='dash'),
            hovertemplate=f'FY %{{x}}<br>{model}: $%{{y:,.0f}}<extra></extra>'))
    verdict = forecast['GL verdict'].iloc[0]
    fig.add_annotation(x=0, y=1.05, xref='paper', yref='paper', text=escape(verdict),
                       showarrow=False, xanchor='left', font=dict(size=11))
    fig.add_vline(x=LAST_COMPLETE_FY + 0.5, line_dash='dot', line_color='gray')
    fig.update_layout(title=f'GL {gl}: {escape(str(actual["description"].iloc[-1]))}',
        template='plotly_white', height=550, hovermode='x unified',
        xaxis=dict(title='Fiscal year (FY)', tickformat='d', dtick=2,
                   range=[int(actual['fy'].min()) - 0.5, int(future_fys[-1]) + 0.5]),
        yaxis=dict(title='Revenue ($)', tickprefix='$', tickformat='~s'),
        legend=dict(orientation='h', y=-0.2, groupclick='togglegroup'), margin=dict(t=110, b=120))
    return fig


## 13. Export a chart for every GL
HTML charts share a local Plotly library and work offline. The index lists every GL and its verdict.

In [ ]:
chart_links = []
for gl in all_gl_codes:
    figure = plot_gl_forecast(gl)
    filename = f'gl_{gl}_forecast.html'
    figure.write_html(plot_dir / filename, include_plotlyjs='directory')
    row = gl_verdict_table.loc[gl_verdict_table['GL Number'].eq(gl)].iloc[0]
    chart_links.append(f'<li><a href="{filename}">GL {gl}: {escape(str(row["Description"]))}</a>'
                       f' - {escape(row["GL verdict"])}</li>')
(plot_dir / 'index.html').write_text(
    '<!doctype html><html><head><meta charset="utf-8"><title>All GL forecasts</title></head>'
    '<body><h1>All GL historical revenue and forecasts</h1><ul>' + ''.join(chart_links) + '</ul></body></html>',
    encoding='utf-8')
print(f'Saved {len(all_gl_codes)} interactive charts: {plot_dir / "index.html"}')


### GL 3001: historical revenue and forecast

In [ ]:
fig_gl_3001 = plot_gl_forecast(3001)
fig_gl_3001.show()


### GL 3002: historical revenue and forecast

In [ ]:
fig_gl_3002 = plot_gl_forecast(3002)
fig_gl_3002.show()


### GL 3003: historical revenue and forecast

In [ ]:
fig_gl_3003 = plot_gl_forecast(3003)
fig_gl_3003.show()


### GL 3004: historical revenue and forecast

In [ ]:
fig_gl_3004 = plot_gl_forecast(3004)
fig_gl_3004.show()


### GL 3005: historical revenue and forecast

In [ ]:
fig_gl_3005 = plot_gl_forecast(3005)
fig_gl_3005.show()


### GL 3028: historical revenue and forecast

In [ ]:
fig_gl_3028 = plot_gl_forecast(3028)
fig_gl_3028.show()


### GL 3029: historical revenue and forecast

In [ ]:
fig_gl_3029 = plot_gl_forecast(3029)
fig_gl_3029.show()


### GL 3030: historical revenue and forecast

In [ ]:
fig_gl_3030 = plot_gl_forecast(3030)
fig_gl_3030.show()


### GL 3031: historical revenue and forecast

In [ ]:
fig_gl_3031 = plot_gl_forecast(3031)
fig_gl_3031.show()


### GL 3032: historical revenue and forecast

In [ ]:
fig_gl_3032 = plot_gl_forecast(3032)
fig_gl_3032.show()


### GL 3033: historical revenue and forecast

In [ ]:
fig_gl_3033 = plot_gl_forecast(3033)
fig_gl_3033.show()


### GL 3034: historical revenue and forecast

In [ ]:
fig_gl_3034 = plot_gl_forecast(3034)
fig_gl_3034.show()


### GL 3035: historical revenue and forecast

In [ ]:
fig_gl_3035 = plot_gl_forecast(3035)
fig_gl_3035.show()


### GL 3041: historical revenue and forecast

In [ ]:
fig_gl_3041 = plot_gl_forecast(3041)
fig_gl_3041.show()


### GL 3042: historical revenue and forecast

In [ ]:
fig_gl_3042 = plot_gl_forecast(3042)
fig_gl_3042.show()


### GL 3043: historical revenue and forecast

In [ ]:
fig_gl_3043 = plot_gl_forecast(3043)
fig_gl_3043.show()


### GL 3044: historical revenue and forecast

In [ ]:
fig_gl_3044 = plot_gl_forecast(3044)
fig_gl_3044.show()


### GL 3045: historical revenue and forecast

In [ ]:
fig_gl_3045 = plot_gl_forecast(3045)
fig_gl_3045.show()


### GL 3046: historical revenue and forecast

In [ ]:
fig_gl_3046 = plot_gl_forecast(3046)
fig_gl_3046.show()


### GL 3047: historical revenue and forecast

In [ ]:
fig_gl_3047 = plot_gl_forecast(3047)
fig_gl_3047.show()


### GL 3048: historical revenue and forecast

In [ ]:
fig_gl_3048 = plot_gl_forecast(3048)
fig_gl_3048.show()


### GL 3049: historical revenue and forecast

In [ ]:
fig_gl_3049 = plot_gl_forecast(3049)
fig_gl_3049.show()


### GL 3050: historical revenue and forecast

In [ ]:
fig_gl_3050 = plot_gl_forecast(3050)
fig_gl_3050.show()


### GL 3051: historical revenue and forecast

In [ ]:
fig_gl_3051 = plot_gl_forecast(3051)
fig_gl_3051.show()


### GL 3052: historical revenue and forecast

In [ ]:
fig_gl_3052 = plot_gl_forecast(3052)
fig_gl_3052.show()


### GL 3053: historical revenue and forecast

In [ ]:
fig_gl_3053 = plot_gl_forecast(3053)
fig_gl_3053.show()


### GL 3055: historical revenue and forecast

In [ ]:
fig_gl_3055 = plot_gl_forecast(3055)
fig_gl_3055.show()


### GL 3057: historical revenue and forecast

In [ ]:
fig_gl_3057 = plot_gl_forecast(3057)
fig_gl_3057.show()


### GL 3058: historical revenue and forecast

In [ ]:
fig_gl_3058 = plot_gl_forecast(3058)
fig_gl_3058.show()


### GL 3059: historical revenue and forecast

In [ ]:
fig_gl_3059 = plot_gl_forecast(3059)
fig_gl_3059.show()


### GL 3060: historical revenue and forecast

In [ ]:
fig_gl_3060 = plot_gl_forecast(3060)
fig_gl_3060.show()


### GL 3061: historical revenue and forecast

In [ ]:
fig_gl_3061 = plot_gl_forecast(3061)
fig_gl_3061.show()


### GL 3062: historical revenue and forecast

In [ ]:
fig_gl_3062 = plot_gl_forecast(3062)
fig_gl_3062.show()


### GL 3064: historical revenue and forecast

In [ ]:
fig_gl_3064 = plot_gl_forecast(3064)
fig_gl_3064.show()


### GL 3065: historical revenue and forecast

In [ ]:
fig_gl_3065 = plot_gl_forecast(3065)
fig_gl_3065.show()


### GL 3066: historical revenue and forecast

In [ ]:
fig_gl_3066 = plot_gl_forecast(3066)
fig_gl_3066.show()


### GL 3067: historical revenue and forecast

In [ ]:
fig_gl_3067 = plot_gl_forecast(3067)
fig_gl_3067.show()


### GL 3068: historical revenue and forecast

In [ ]:
fig_gl_3068 = plot_gl_forecast(3068)
fig_gl_3068.show()


### GL 3069: historical revenue and forecast

In [ ]:
fig_gl_3069 = plot_gl_forecast(3069)
fig_gl_3069.show()


### GL 3070: historical revenue and forecast

In [ ]:
fig_gl_3070 = plot_gl_forecast(3070)
fig_gl_3070.show()


### GL 3071: historical revenue and forecast

In [ ]:
fig_gl_3071 = plot_gl_forecast(3071)
fig_gl_3071.show()


### GL 3072: historical revenue and forecast

In [ ]:
fig_gl_3072 = plot_gl_forecast(3072)
fig_gl_3072.show()


### GL 3073: historical revenue and forecast

In [ ]:
fig_gl_3073 = plot_gl_forecast(3073)
fig_gl_3073.show()


### GL 3074: historical revenue and forecast

In [ ]:
fig_gl_3074 = plot_gl_forecast(3074)
fig_gl_3074.show()


### GL 3075: historical revenue and forecast

In [ ]:
fig_gl_3075 = plot_gl_forecast(3075)
fig_gl_3075.show()


### GL 3100: historical revenue and forecast

In [ ]:
fig_gl_3100 = plot_gl_forecast(3100)
fig_gl_3100.show()


### GL 3101: historical revenue and forecast

In [ ]:
fig_gl_3101 = plot_gl_forecast(3101)
fig_gl_3101.show()


### GL 3102: historical revenue and forecast

In [ ]:
fig_gl_3102 = plot_gl_forecast(3102)
fig_gl_3102.show()


### GL 3103: historical revenue and forecast

In [ ]:
fig_gl_3103 = plot_gl_forecast(3103)
fig_gl_3103.show()


### GL 3105: historical revenue and forecast

In [ ]:
fig_gl_3105 = plot_gl_forecast(3105)
fig_gl_3105.show()


### GL 3106: historical revenue and forecast

In [ ]:
fig_gl_3106 = plot_gl_forecast(3106)
fig_gl_3106.show()


### GL 3107: historical revenue and forecast

In [ ]:
fig_gl_3107 = plot_gl_forecast(3107)
fig_gl_3107.show()


### GL 3109: historical revenue and forecast

In [ ]:
fig_gl_3109 = plot_gl_forecast(3109)
fig_gl_3109.show()


### GL 3110: historical revenue and forecast

In [ ]:
fig_gl_3110 = plot_gl_forecast(3110)
fig_gl_3110.show()


### GL 3113: historical revenue and forecast

In [ ]:
fig_gl_3113 = plot_gl_forecast(3113)
fig_gl_3113.show()


### GL 3114: historical revenue and forecast

In [ ]:
fig_gl_3114 = plot_gl_forecast(3114)
fig_gl_3114.show()


### GL 3115: historical revenue and forecast

In [ ]:
fig_gl_3115 = plot_gl_forecast(3115)
fig_gl_3115.show()


### GL 3116: historical revenue and forecast

In [ ]:
fig_gl_3116 = plot_gl_forecast(3116)
fig_gl_3116.show()


### GL 3120: historical revenue and forecast

In [ ]:
fig_gl_3120 = plot_gl_forecast(3120)
fig_gl_3120.show()


### GL 3121: historical revenue and forecast

In [ ]:
fig_gl_3121 = plot_gl_forecast(3121)
fig_gl_3121.show()


### GL 3129: historical revenue and forecast

In [ ]:
fig_gl_3129 = plot_gl_forecast(3129)
fig_gl_3129.show()


### GL 3130: historical revenue and forecast

In [ ]:
fig_gl_3130 = plot_gl_forecast(3130)
fig_gl_3130.show()


### GL 3131: historical revenue and forecast

In [ ]:
fig_gl_3131 = plot_gl_forecast(3131)
fig_gl_3131.show()


### GL 3135: historical revenue and forecast

In [ ]:
fig_gl_3135 = plot_gl_forecast(3135)
fig_gl_3135.show()


### GL 3143: historical revenue and forecast

In [ ]:
fig_gl_3143 = plot_gl_forecast(3143)
fig_gl_3143.show()


### GL 3150: historical revenue and forecast

In [ ]:
fig_gl_3150 = plot_gl_forecast(3150)
fig_gl_3150.show()


### GL 3151: historical revenue and forecast

In [ ]:
fig_gl_3151 = plot_gl_forecast(3151)
fig_gl_3151.show()


### GL 3152: historical revenue and forecast

In [ ]:
fig_gl_3152 = plot_gl_forecast(3152)
fig_gl_3152.show()


### GL 3161: historical revenue and forecast

In [ ]:
fig_gl_3161 = plot_gl_forecast(3161)
fig_gl_3161.show()


### GL 3162: historical revenue and forecast

In [ ]:
fig_gl_3162 = plot_gl_forecast(3162)
fig_gl_3162.show()


### GL 3163: historical revenue and forecast

In [ ]:
fig_gl_3163 = plot_gl_forecast(3163)
fig_gl_3163.show()


### GL 3164: historical revenue and forecast

In [ ]:
fig_gl_3164 = plot_gl_forecast(3164)
fig_gl_3164.show()


### GL 3165: historical revenue and forecast

In [ ]:
fig_gl_3165 = plot_gl_forecast(3165)
fig_gl_3165.show()


### GL 3166: historical revenue and forecast

In [ ]:
fig_gl_3166 = plot_gl_forecast(3166)
fig_gl_3166.show()


### GL 3167: historical revenue and forecast

In [ ]:
fig_gl_3167 = plot_gl_forecast(3167)
fig_gl_3167.show()


### GL 3168: historical revenue and forecast

In [ ]:
fig_gl_3168 = plot_gl_forecast(3168)
fig_gl_3168.show()


### GL 3169: historical revenue and forecast

In [ ]:
fig_gl_3169 = plot_gl_forecast(3169)
fig_gl_3169.show()


### GL 3170: historical revenue and forecast

In [ ]:
fig_gl_3170 = plot_gl_forecast(3170)
fig_gl_3170.show()


### GL 3171: historical revenue and forecast

In [ ]:
fig_gl_3171 = plot_gl_forecast(3171)
fig_gl_3171.show()


### GL 3172: historical revenue and forecast

In [ ]:
fig_gl_3172 = plot_gl_forecast(3172)
fig_gl_3172.show()


### GL 3173: historical revenue and forecast

In [ ]:
fig_gl_3173 = plot_gl_forecast(3173)
fig_gl_3173.show()


### GL 3174: historical revenue and forecast

In [ ]:
fig_gl_3174 = plot_gl_forecast(3174)
fig_gl_3174.show()


### GL 3175: historical revenue and forecast

In [ ]:
fig_gl_3175 = plot_gl_forecast(3175)
fig_gl_3175.show()


### GL 3177: historical revenue and forecast

In [ ]:
fig_gl_3177 = plot_gl_forecast(3177)
fig_gl_3177.show()


### GL 3178: historical revenue and forecast

In [ ]:
fig_gl_3178 = plot_gl_forecast(3178)
fig_gl_3178.show()


### GL 3179: historical revenue and forecast

In [ ]:
fig_gl_3179 = plot_gl_forecast(3179)
fig_gl_3179.show()


### GL 3180: historical revenue and forecast

In [ ]:
fig_gl_3180 = plot_gl_forecast(3180)
fig_gl_3180.show()


### GL 3181: historical revenue and forecast

In [ ]:
fig_gl_3181 = plot_gl_forecast(3181)
fig_gl_3181.show()


### GL 3190: historical revenue and forecast

In [ ]:
fig_gl_3190 = plot_gl_forecast(3190)
fig_gl_3190.show()


### GL 3200: historical revenue and forecast

In [ ]:
fig_gl_3200 = plot_gl_forecast(3200)
fig_gl_3200.show()


### GL 3202: historical revenue and forecast

In [ ]:
fig_gl_3202 = plot_gl_forecast(3202)
fig_gl_3202.show()


### GL 3203: historical revenue and forecast

In [ ]:
fig_gl_3203 = plot_gl_forecast(3203)
fig_gl_3203.show()


### GL 3204: historical revenue and forecast

In [ ]:
fig_gl_3204 = plot_gl_forecast(3204)
fig_gl_3204.show()


### GL 3205: historical revenue and forecast

In [ ]:
fig_gl_3205 = plot_gl_forecast(3205)
fig_gl_3205.show()


### GL 3206: historical revenue and forecast

In [ ]:
fig_gl_3206 = plot_gl_forecast(3206)
fig_gl_3206.show()


### GL 3208: historical revenue and forecast

In [ ]:
fig_gl_3208 = plot_gl_forecast(3208)
fig_gl_3208.show()


### GL 3220: historical revenue and forecast

In [ ]:
fig_gl_3220 = plot_gl_forecast(3220)
fig_gl_3220.show()


### GL 3222: historical revenue and forecast

In [ ]:
fig_gl_3222 = plot_gl_forecast(3222)
fig_gl_3222.show()


### GL 3240: historical revenue and forecast

In [ ]:
fig_gl_3240 = plot_gl_forecast(3240)
fig_gl_3240.show()


### GL 3241: historical revenue and forecast

In [ ]:
fig_gl_3241 = plot_gl_forecast(3241)
fig_gl_3241.show()


### GL 3242: historical revenue and forecast

In [ ]:
fig_gl_3242 = plot_gl_forecast(3242)
fig_gl_3242.show()


### GL 3243: historical revenue and forecast

In [ ]:
fig_gl_3243 = plot_gl_forecast(3243)
fig_gl_3243.show()


### GL 3244: historical revenue and forecast

In [ ]:
fig_gl_3244 = plot_gl_forecast(3244)
fig_gl_3244.show()


### GL 3245: historical revenue and forecast

In [ ]:
fig_gl_3245 = plot_gl_forecast(3245)
fig_gl_3245.show()


### GL 3247: historical revenue and forecast

In [ ]:
fig_gl_3247 = plot_gl_forecast(3247)
fig_gl_3247.show()


### GL 3248: historical revenue and forecast

In [ ]:
fig_gl_3248 = plot_gl_forecast(3248)
fig_gl_3248.show()


### GL 3250: historical revenue and forecast

In [ ]:
fig_gl_3250 = plot_gl_forecast(3250)
fig_gl_3250.show()


### GL 3254: historical revenue and forecast

In [ ]:
fig_gl_3254 = plot_gl_forecast(3254)
fig_gl_3254.show()


### GL 3255: historical revenue and forecast

In [ ]:
fig_gl_3255 = plot_gl_forecast(3255)
fig_gl_3255.show()


### GL 3271: historical revenue and forecast

In [ ]:
fig_gl_3271 = plot_gl_forecast(3271)
fig_gl_3271.show()


### GL 3272: historical revenue and forecast

In [ ]:
fig_gl_3272 = plot_gl_forecast(3272)
fig_gl_3272.show()


### GL 3274: historical revenue and forecast

In [ ]:
fig_gl_3274 = plot_gl_forecast(3274)
fig_gl_3274.show()


### GL 3275: historical revenue and forecast

In [ ]:
fig_gl_3275 = plot_gl_forecast(3275)
fig_gl_3275.show()


### GL 3276: historical revenue and forecast

In [ ]:
fig_gl_3276 = plot_gl_forecast(3276)
fig_gl_3276.show()


### GL 3290: historical revenue and forecast

In [ ]:
fig_gl_3290 = plot_gl_forecast(3290)
fig_gl_3290.show()


### GL 3291: historical revenue and forecast

In [ ]:
fig_gl_3291 = plot_gl_forecast(3291)
fig_gl_3291.show()


### GL 3601: historical revenue and forecast

In [ ]:
fig_gl_3601 = plot_gl_forecast(3601)
fig_gl_3601.show()


### GL 3756: historical revenue and forecast

In [ ]:
fig_gl_3756 = plot_gl_forecast(3756)
fig_gl_3756.show()


### GL 3776: historical revenue and forecast

In [ ]:
fig_gl_3776 = plot_gl_forecast(3776)
fig_gl_3776.show()


### GL 4021: historical revenue and forecast

In [ ]:
fig_gl_4021 = plot_gl_forecast(4021)
fig_gl_4021.show()


### GL 4102: historical revenue and forecast

In [ ]:
fig_gl_4102 = plot_gl_forecast(4102)
fig_gl_4102.show()


### GL 4117: historical revenue and forecast

In [ ]:
fig_gl_4117 = plot_gl_forecast(4117)
fig_gl_4117.show()


### GL 4203: historical revenue and forecast

In [ ]:
fig_gl_4203 = plot_gl_forecast(4203)
fig_gl_4203.show()


### GL 4209: historical revenue and forecast

In [ ]:
fig_gl_4209 = plot_gl_forecast(4209)
fig_gl_4209.show()


### GL 4250: historical revenue and forecast

In [ ]:
fig_gl_4250 = plot_gl_forecast(4250)
fig_gl_4250.show()


### GL 4251: historical revenue and forecast

In [ ]:
fig_gl_4251 = plot_gl_forecast(4251)
fig_gl_4251.show()


### GL 4253: historical revenue and forecast

In [ ]:
fig_gl_4253 = plot_gl_forecast(4253)
fig_gl_4253.show()


### GL 4254: historical revenue and forecast

In [ ]:
fig_gl_4254 = plot_gl_forecast(4254)
fig_gl_4254.show()


### GL 4401: historical revenue and forecast

In [ ]:
fig_gl_4401 = plot_gl_forecast(4401)
fig_gl_4401.show()


### GL 4402: historical revenue and forecast

In [ ]:
fig_gl_4402 = plot_gl_forecast(4402)
fig_gl_4402.show()


### GL 4403: historical revenue and forecast

In [ ]:
fig_gl_4403 = plot_gl_forecast(4403)
fig_gl_4403.show()


### GL 4404: historical revenue and forecast

In [ ]:
fig_gl_4404 = plot_gl_forecast(4404)
fig_gl_4404.show()


### GL 4405: historical revenue and forecast

In [ ]:
fig_gl_4405 = plot_gl_forecast(4405)
fig_gl_4405.show()


### GL 4406: historical revenue and forecast

In [ ]:
fig_gl_4406 = plot_gl_forecast(4406)
fig_gl_4406.show()


### GL 4407: historical revenue and forecast

In [ ]:
fig_gl_4407 = plot_gl_forecast(4407)
fig_gl_4407.show()


### GL 4408: historical revenue and forecast

In [ ]:
fig_gl_4408 = plot_gl_forecast(4408)
fig_gl_4408.show()


### GL 4409: historical revenue and forecast

In [ ]:
fig_gl_4409 = plot_gl_forecast(4409)
fig_gl_4409.show()


### GL 4410: historical revenue and forecast

In [ ]:
fig_gl_4410 = plot_gl_forecast(4410)
fig_gl_4410.show()


### GL 4420: historical revenue and forecast

In [ ]:
fig_gl_4420 = plot_gl_forecast(4420)
fig_gl_4420.show()


### GL 4454: historical revenue and forecast

In [ ]:
fig_gl_4454 = plot_gl_forecast(4454)
fig_gl_4454.show()


### GL 4473: historical revenue and forecast

In [ ]:
fig_gl_4473 = plot_gl_forecast(4473)
fig_gl_4473.show()


### GL 4621: historical revenue and forecast

In [ ]:
fig_gl_4621 = plot_gl_forecast(4621)
fig_gl_4621.show()


### GL 4669: historical revenue and forecast

In [ ]:
fig_gl_4669 = plot_gl_forecast(4669)
fig_gl_4669.show()


### GL 4681: historical revenue and forecast

In [ ]:
fig_gl_4681 = plot_gl_forecast(4681)
fig_gl_4681.show()


### GL 4694: historical revenue and forecast

In [ ]:
fig_gl_4694 = plot_gl_forecast(4694)
fig_gl_4694.show()


### GL 4703: historical revenue and forecast

In [ ]:
fig_gl_4703 = plot_gl_forecast(4703)
fig_gl_4703.show()


### GL 4705: historical revenue and forecast

In [ ]:
fig_gl_4705 = plot_gl_forecast(4705)
fig_gl_4705.show()


### GL 4739: historical revenue and forecast

In [ ]:
fig_gl_4739 = plot_gl_forecast(4739)
fig_gl_4739.show()


### GL 4741: historical revenue and forecast

In [ ]:
fig_gl_4741 = plot_gl_forecast(4741)
fig_gl_4741.show()


### GL 4774: historical revenue and forecast

In [ ]:
fig_gl_4774 = plot_gl_forecast(4774)
fig_gl_4774.show()


### GL 4790: historical revenue and forecast

In [ ]:
fig_gl_4790 = plot_gl_forecast(4790)
fig_gl_4790.show()


### GL 4791: historical revenue and forecast

In [ ]:
fig_gl_4791 = plot_gl_forecast(4791)
fig_gl_4791.show()


### GL 4792: historical revenue and forecast

In [ ]:
fig_gl_4792 = plot_gl_forecast(4792)
fig_gl_4792.show()


### GL 4793: historical revenue and forecast

In [ ]:
fig_gl_4793 = plot_gl_forecast(4793)
fig_gl_4793.show()


### GL 4794: historical revenue and forecast

In [ ]:
fig_gl_4794 = plot_gl_forecast(4794)
fig_gl_4794.show()


### GL 4861: historical revenue and forecast

In [ ]:
fig_gl_4861 = plot_gl_forecast(4861)
fig_gl_4861.show()


### GL 4862: historical revenue and forecast

In [ ]:
fig_gl_4862 = plot_gl_forecast(4862)
fig_gl_4862.show()


### GL 4863: historical revenue and forecast

In [ ]:
fig_gl_4863 = plot_gl_forecast(4863)
fig_gl_4863.show()


### GL 4864: historical revenue and forecast

In [ ]:
fig_gl_4864 = plot_gl_forecast(4864)
fig_gl_4864.show()


### GL 4865: historical revenue and forecast

In [ ]:
fig_gl_4865 = plot_gl_forecast(4865)
fig_gl_4865.show()


### GL 4866: historical revenue and forecast

In [ ]:
fig_gl_4866 = plot_gl_forecast(4866)
fig_gl_4866.show()


### GL 4867: historical revenue and forecast

In [ ]:
fig_gl_4867 = plot_gl_forecast(4867)
fig_gl_4867.show()


### GL 4868: historical revenue and forecast

In [ ]:
fig_gl_4868 = plot_gl_forecast(4868)
fig_gl_4868.show()
